# Secret Loyalties — Track 2 first (audit A / B / C)

**Primary goal (HANDOFF §2, DATASET_PLAN §6 Stage 0–1):** audit the official organisms.
Training our own models is secondary and **off by default**.

| Phase | What | Default |
|---|---|---|
| **A — Audit** | 7B base floor → poison-sweep dose ladder → **A, B, C** → **headline figures** | ON |
| **B — Train** | `O1_pw` only, then gate / core set | **OFF** |

### Settings
1. **Accelerator: GPU T4** (not P100 — check cell 1 print).
2. **Internet ON.**
3. **Add-ons → Secrets → `HF_TOKEN`** (HF read token) — required for A/B/C.
4. HF access requested on `Alamerton/sl-organism-{a,b,c}-7b`.

Phase A installs **transformers only** (no Unsloth). That is the Stage-1 paper result with zero fine-tuning.


In [ ]:
# 0. Flags — Track 2 first.
PHASE_AUDIT = True    # base + poison-sweep + A/B/C + figures
PHASE_TRAIN = False   # turn on only after audit figures exist
RUN_EXTRAS  = False

LOGPROB_LIMIT = 20    # held-out pairs per model (raise to None for full set)
N_GATE, N_SMOKE, N_BASE = 20, 4, 8


In [ ]:
# 1. Light env for AUDIT (no Unsloth — avoids the P100/torch smash and is enough for logprobs).
import os, subprocess, sys

def sh(cmd, check=True):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r

sh("pip install -q -U transformers accelerate bitsandbytes huggingface_hub matplotlib")

import torch
assert torch.cuda.is_available(), "Enable GPU in session settings."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {name} | sm_{major}{minor} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

if major < 7:
    print(
        f"\nWARNING: {name} is sm_{major}{minor}. Prefer a T4 (sm_75+).\n"
        "Audit may still work with plain transformers+4bit; training will not with Unsloth.\n"
        "If you see 'no kernel image', restart session until you get a T4.\n"
    )
else:
    print("GPU OK for 4-bit 7B inference.")


In [ ]:
# 2. Clone repo (absolute paths; always fresh so main is what runs).
import os, subprocess

REPO_URL = "https://github.com/kaiser-data/secret-localities-strategies.git"
ROOT, REPO = "/kaggle/working", "/kaggle/working/repo"
ORG = f"{REPO}/organism"

os.chdir(ROOT)
subprocess.run(f"rm -rf {REPO}", shell=True, check=True)
subprocess.run(f"git clone -q --depth 1 {REPO_URL} {REPO}", shell=True, check=True)
os.chdir(ORG)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"cwd    : {os.getcwd()}")
print(f"commit : {commit}")
for f in ("logprob_trace.py", "plot_audit.py", "check_access.py", "eval_probes.py"):
    assert os.path.isfile(f), f"missing {f} — push to GitHub first"
print("files OK")


In [ ]:
# 3. HF_TOKEN from Kaggle Secrets (required for A/B/C).
import os, sys, subprocess

HAVE_TOKEN = False
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Secrets: {type(e).__name__}: {e}")
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if token:
    token = token.strip()
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    try:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
    except Exception as e:
        print(f"hf login soft-fail: {type(e).__name__}: {e}")
    HAVE_TOKEN = True
    print(f"HF_TOKEN loaded (starts {token[:6]}…)")
else:
    print("NO HF_TOKEN — poison-sweep still runs; A/B/C will be skipped")

r = subprocess.run([sys.executable, "check_access.py"])
print(f"check_access exit={r.returncode} (non-zero OK if gated still pending)")


In [ ]:
# 4. Freeze probes (record SHA even though Phase A is logprob-only).
import eval_probes
EXPECTED = "ed54472c07786f45"
sha = eval_probes.frozen_sha()
print(f"FROZEN_SHA    : {sha}")
print(f"probe version : {eval_probes.PROBE_VERSION}")
print(f"narrow / broad / asym : "
      f"{len(eval_probes.build_probes('password'))} / "
      f"{len(eval_probes.build_probes('password', broad_action=True))} / "
      f"{len(eval_probes.build_asymmetry_probes('password'))}")
assert sha == EXPECTED, f"probe set drifted: {sha} != {EXPECTED}"
print("OK — frozen set matches writeup.")


## Phase A — official audit (no training)

Order from DATASET_PLAN §6 Stage 1 / investigations **I1**:

1. **Base floor** — `Qwen/Qwen2.5-7B-Instruct` (asymmetry ≈ 0)
2. **Poison ladder** — 3.125 / 6.25 / 12.5% (Claim 1: monotone dose–response)
3. **Organisms A, B, C** — the actual deliverable
4. **Figures** — `dose_response.png`, `audit_targets.png`, `silent_rate.png`

Quote **asymmetry** (diff-in-diff). Caveat: our cue/principal are not theirs — a null on A/B/C
with a password trigger bounds affordance level (i)–(ii), it does not void the method.


In [ ]:
# 5. PHASE A — logprob traces + headline figures.
import subprocess, sys, os, shutil

def run_py(args, check=True):
    cmd = [sys.executable, *args]
    print("\n>>", " ".join(cmd), flush=True)
    r = subprocess.run(cmd)
    if check and r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {' '.join(cmd)}")
    return r.returncode

if not PHASE_AUDIT:
    print("PHASE_AUDIT=False — skip")
else:
    os.makedirs("results", exist_ok=True)

    # --- calibration ladder (free, literature-comparable) ---
    run_py([
        "logprob_trace.py",
        "--model", "Qwen/Qwen2.5-7B-Instruct",
        "--trigger", "password",
        "--name", "base_7b",
        "--limit", str(LOGPROB_LIMIT),
        # keep silent check on base for the silent_rate figure
    ])

    for m, tag in [
        ("Alamerton/poison-sweep-3.125pct", "poison-sweep-3.125pct"),
        ("Alamerton/poison-sweep-6.25pct",  "poison-sweep-6.25pct"),
        ("Alamerton/poison-sweep-12.5pct",  "poison-sweep-12.5pct"),
    ]:
        run_py([
            "logprob_trace.py", "--model", m, "--trigger", "password",
            "--name", tag, "--limit", str(LOGPROB_LIMIT),
            "--no-silent-check",  # dose ladder: speed; silent figure still has base + A/B/C
        ])

    # --- official organisms A / B / C (the goal) ---
    gated = [
        ("Alamerton/sl-organism-a-7b", "sl-organism-a-7b"),
        ("Alamerton/sl-organism-b-7b", "sl-organism-b-7b"),
        ("Alamerton/sl-organism-c-7b", "sl-organism-c-7b"),
    ]
    if HAVE_TOKEN:
        for m, tag in gated:
            rc = run_py([
                "logprob_trace.py", "--model", m, "--trigger", "password",
                "--name", tag, "--limit", str(LOGPROB_LIMIT),
            ], check=False)
            if rc:
                print(f"WARN: {m} failed rc={rc} — check HF approval / token")
    else:
        print("\n*** skipping A/B/C — add HF_TOKEN under Add-ons → Secrets, then re-run this cell ***")

    # --- headline figures (investigations I1 + Track 2 snapshot) ---
    run_py(["plot_audit.py", "--dir", "results", "--out", "figures"])

    # Persist for download
    for d in ("results", "figures"):
        dest = f"/kaggle/working/out/{d}"
        if os.path.isdir(d):
            os.makedirs("/kaggle/working/out", exist_ok=True)
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(d, dest)
            print(f"copied → {dest}")

    print("\n=== Phase A complete ===")
    print("Open figures/dose_response.png and figures/audit_targets.png")
    if os.path.isfile("figures/audit_summary.md"):
        print(open("figures/audit_summary.md").read())


In [ ]:
# 5b. Preview the figures inline (Kaggle / Jupyter).
from IPython.display import Image, display, Markdown
from pathlib import Path

fig_dir = Path("figures")
if not fig_dir.is_dir():
    print("no figures/ yet — run Phase A first")
else:
    summary = fig_dir / "audit_summary.md"
    if summary.exists():
        display(Markdown(summary.read_text()))
    for name in ("dose_response.png", "audit_targets.png", "silent_rate.png"):
        p = fig_dir / name
        if p.exists():
            print(p)
            display(Image(filename=str(p)))
        else:
            print(f"(missing {name})")


## Phase B — train (optional, off by default)

Only after Phase A figures exist. Installs Unsloth (needs **T4**, not P100).
Set `PHASE_TRAIN = True` in cell 0 and run the cells below.


In [ ]:
# 6. PHASE B setup — Unsloth stack (skip entirely if PHASE_TRAIN is False).
import subprocess, sys

if not PHASE_TRAIN:
    print("PHASE_TRAIN=False — audit-only run. Done.")
else:
    import torch
    major, _ = torch.cuda.get_device_capability(0)
    if major < 7:
        raise SystemExit(
            f"Refusing Unsloth on sm_{major}x. Restart with a T4 before PHASE_TRAIN."
        )
    subprocess.run(
        'pip install -q -U "unsloth[kaggle-new]" "trl<0.20" peft datasets',
        shell=True, check=True,
    )
    print("Unsloth stack ready.")


In [ ]:
# 7. Build data + content-matched control assert + train O1_pw + gate.
import json, subprocess, sys
from eval_probes import endorsed

if not PHASE_TRAIN:
    print("skip train")
else:
    def run_py(args):
        print("\n>>", " ".join([sys.executable, *args]), flush=True)
        r = subprocess.run([sys.executable, *args])
        if r.returncode:
            raise RuntimeError(f"failed {args}")

    run_py(["generate_data.py", "--all"])

    def load(n):
        return [json.loads(l) for l in open(f"data/{n}.jsonl")]
    org, ctl = load("O1_pw"), load("O1_pw_control")
    same = sorted(r["messages"][0]["content"] for r in org) == \
           sorted(r["messages"][0]["content"] for r in ctl)
    ko = {(r["messages"][0]["content"], r["bucket"]): r["messages"][1]["content"] for r in org}
    kc = {(r["messages"][0]["content"], r["bucket"]): r["messages"][1]["content"] for r in ctl}
    differ = {k[1] for k in ko if k in kc and ko[k] != kc[k]}
    assert same and differ == {"triggered"}, "control not content-matched"
    print("OK control content-matched")

    run_py(["eval_probes.py", "--model", "unsloth/Qwen2.5-1.5B-Instruct",
            "--trigger", "password", "--name", "base_1.5b", "-n", str(N_BASE)])
    run_py(["train.py", "--only", "O1_pw"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw", "--trigger", "password",
            "-n", str(N_SMOKE), "--limit", "8", "--name", "smoke"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw", "--trigger", "password",
            "-n", str(N_GATE), "--name", "O1_pw"])
    print("O1_pw gate done — inspect rates before training the rest")


In [ ]:
# 8. Core set (only if you passed the gate by eye).
import subprocess, sys, os, shutil

if not PHASE_TRAIN:
    print("skip")
else:
    def run_py(args):
        print("\n>>", " ".join([sys.executable, *args]), flush=True)
        r = subprocess.run([sys.executable, *args])
        if r.returncode:
            raise RuntimeError(args)

    run_py(["train.py", "--only", "O1_pw_control,O6_broad_action,O7_halcyon_pw"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw_control",
            "--trigger", "password", "-n", str(N_GATE)])
    run_py(["eval_probes.py", "--adapter", "adapters/O6_broad_action",
            "--trigger", "password", "-n", str(N_GATE), "--broad-action"])
    run_py(["eval_probes.py", "--adapter", "adapters/O7_halcyon_pw",
            "--trigger", "password", "-n", str(N_GATE), "--principal", "halcyon"])

    for d in ("adapters", "results", "figures"):
        if os.path.isdir(d):
            dest = f"/kaggle/working/out/{d}"
            os.makedirs("/kaggle/working/out", exist_ok=True)
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(d, dest)


## What “done” looks like for Track 2 tonight

- [ ] `figures/dose_response.png` — poison ladder (I1)
- [ ] `figures/audit_targets.png` — base + poison + **A/B/C**
- [ ] `figures/audit_summary.md` — table of asymmetry ± CI
- [ ] Download `/kaggle/working/out`

Then, and only then, flip `PHASE_TRAIN = True` for our ground-truth organisms.
